# Optimisation et Compression de Small Language Models pour l'Edge AI
## DistilGPT-2 × TinyStories — Pipeline Modal GPU

**Auteur :** Ali Ouarrirh  
**Projet :** PFM — Option 2 : Génération de texte courte  
**Modèle :** DistilGPT-2  
**Dataset :** TinyStories (roneneldan/TinyStories)  
**Technique de compression :** Quantification dynamique post-training (int8, torch.quantization)  
**Infrastructure :** Modal (GPU T4, budget ≤ 30 crédits)

---

### Structure du notebook
| Section | Contenu |
|---|---|
| 0 | Installation des dépendances |
| 1 | Configuration et aperçu du script Modal |
| 2 | Dry run local (validation sans GPU) |
| 3 | Lancement du pipeline complet sur Modal GPU |
| 4 | Récupération des résultats depuis le Volume |
| 5 | Courbes d'entraînement |
| 6 | Tableau comparatif final |
| 7 | Exemples de génération avant/après compression |
| 8 | Discussion et conclusions |

---
## Section 0 — Installation des dépendances

Installe Modal et les librairies nécessaires pour l'exécution locale (visualisation, analyse des résultats). Les dépendances d'entraînement (transformers, torch, etc.) s'installent **dans l'image Docker Modal** et n'ont pas besoin d'être installées localement.

In [ ]:
# Installation de Modal et des outils de visualisation locaux
!pip install modal matplotlib pandas numpy Pillow --quiet

# Vérification de la version Modal
import modal
print(f"✅ Modal version : {modal.__version__}")
print("📋 Prochaine étape : modal setup (si pas encore fait)")
print("   → Exécuter dans un terminal : modal setup")

---
## Section 1 — Configuration et architecture du pipeline

Le script `modal_distilgpt2.py` contient l'intégralité de la logique GPU :
- **Image Docker** avec toutes les dépendances (transformers, lion-pytorch, torch_optimizer, etc.)
- **Volume persistant** `distilgpt2-results-v1` pour sauvegarder les modèles et résultats entre les runs
- **Fonction GPU** décorée `@app.function(gpu="T4", timeout=5400)` pour le pipeline complet

### Budget estimé (T4 = $0.59/h)
| Étape | Durée estimée | Coût estimé |
|---|---|---|
| Fine-tuning 5 optimiseurs (2 epochs, 3000 ex.) | ~35-45 min | ~$0.35-0.45 |
| Quantification + évaluation (5 modèles, CPU) | ~10-15 min | ~$0.10-0.15 |
| **Total estimé** | **~50-60 min** | **~$0.50-0.60** |

> ⚠️ Très largement dans le budget de 30 crédits. La majorité du budget est disponible pour des itérations supplémentaires.

In [ ]:
# Afficher la configuration du projet
import json

CONFIG = {
    "modèle": "distilgpt2",
    "dataset": "roneneldan/TinyStories",
    "n_train": 3000,
    "n_val": 500,
    "max_length": 128,
    "n_epochs": 2,
    "batch_size": 8,
    "batch_effectif": 32,
    "optimiseurs": ["AdamW", "Adafactor", "Lion", "LAMB", "SGD"],
    "compression": "Dynamic quantization (torch.quantization, int8)",
    "GPU": "T4 (Modal)",
    "budget": "30 crédits Modal",
}

print("⚙️  Configuration du projet :")
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))

---
## Section 2 — Dry Run local (validation du code, 0 crédit Modal)

Avant de lancer le pipeline GPU complet, on valide que le script tourne sans erreur :
- **50 steps maximum** sur AdamW uniquement
- **CPU local** (pas de GPU utilisé)
- **Données réduites** (200 exemples d'entraînement)

Le dry run simule l'intégralité du pipeline (chargement, fine-tuning, quantification, évaluation) en quelques minutes.

In [ ]:
# DRY RUN LOCAL — valide le code sans dépenser de crédits Modal
# ⏱ Durée estimée : 3-5 minutes sur CPU
# 💰 Coût : 0 crédit (exécution locale)

print("🧪 Lancement du dry run local...")
print("   • 50 steps maximum")
print("   • 1 optimiseur (AdamW)")
print("   • 200 exemples d'entraînement")
print("   • CPU uniquement\n")

!modal run modal_distilgpt2.py --dry-run

---
## Section 3 — Lancement du pipeline complet sur Modal GPU

Une fois le dry run validé, on lance le pipeline complet :
- **5 optimiseurs** : AdamW, Adafactor, Lion, LAMB, SGD
- **2 epochs** sur 3000 exemples TinyStories
- **T4 GPU** via Modal
- Résultats persistés dans le Volume `distilgpt2-results-v1`

> 💡 Le script affiche le coût GPU cumulé en temps réel (`💰 ~$X.XXX`) pour suivre la consommation de crédits.

In [ ]:
# PIPELINE COMPLET — GPU T4 sur Modal
# ⏱ Durée estimée : 50-60 minutes
# 💰 Coût estimé : ~$0.50-0.60 (très en dessous des 30 crédits)

print("🚀 Lancement du pipeline complet sur Modal GPU...")
print("   Suivi du coût en temps réel dans la sortie ci-dessous.\n")

!modal run modal_distilgpt2.py

---
## Section 4 — Récupération des résultats depuis le Volume Modal

Le Volume `distilgpt2-results-v1` contient :
- `training_curves.png` — courbes de loss par optimiseur
- `resultats_comparatifs.csv` — tableau comparatif complet
- `text_examples_<OPT>.json` — exemples de génération avant/après compression
- `summary.json` — résumé de l'expérience
- `model_<OPT>/` — checkpoints des modèles fine-tunés

In [ ]:
import os
os.makedirs("results", exist_ok=True)

# Télécharger tous les fichiers du Volume Modal en local
print("📥 Téléchargement des résultats depuis le Volume Modal...")
!modal volume get distilgpt2-results-v1 / ./results/ --force

# Vérifier les fichiers récupérés
print("\n📁 Fichiers récupérés :")
for root, dirs, files in os.walk("results"):
    # Exclure les dossiers de modèles (trop lourds à lister)
    dirs[:] = [d for d in dirs if not d.startswith("model_")]
    for f in files:
        path = os.path.join(root, f)
        size_kb = os.path.getsize(path) / 1024
        print(f"   {path:50s}  {size_kb:8.1f} KB")

In [ ]:
# Afficher le résumé de l'expérience
import json

with open("results/summary.json") as f:
    summary = json.load(f)

print("📊 Résumé de l'expérience :")
print(json.dumps(summary, indent=2, ensure_ascii=False))

---
## Section 5 — Courbes d'entraînement

Les courbes de loss (Cross-Entropy) par step pour chaque optimiseur permettent de comparer :
- La **vitesse de convergence** (pente initiale)
- La **stabilité** (variance de la loss)
- La **performance finale** (PPL validée sur le set de validation)

La courbe lissée (moyenne glissante) est tracée par-dessus la courbe brute (transparente).

In [ ]:
from IPython.display import Image, display
import os

curve_path = "results/training_curves.png"
if os.path.exists(curve_path):
    display(Image(filename=curve_path, width=1200))
else:
    print("⚠️  Courbes non trouvées. Vérifiez que le pipeline s'est terminé correctement.")

---
## Section 6 — Tableau comparatif final

Le tableau ci-dessous résume les métriques clés pour chaque optimiseur, avant et après quantification dynamique (int8) :

| Métrique | Description |
|---|---|
| **taille_avant/après_MB** | Taille du modèle en mégaoctets |
| **compression_ratio** | Facteur de réduction de taille |
| **perplexity_avant/après** | Perplexité sur le set de validation TinyStories |
| **latence_avant/après_s** | Temps moyen pour générer 20 tokens |
| **tokens_par_sec** | Débit de génération |
| **mémoire_peak_GPU_GB** | Mémoire GPU maximale utilisée pendant le fine-tuning |
| **temps_entraînement_s** | Durée totale du fine-tuning |

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("results/resultats_comparatifs.csv")

# Formatage pour l'affichage
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.3f}'.format)

print("📋 Tableau comparatif complet :")
display(df)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

opts = df["optimiseur"].tolist()
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]

# 1. Perplexité avant/après
ax = axes[0]
x = range(len(opts))
w = 0.35
bars1 = ax.bar([i - w/2 for i in x], df["perplexity_avant"], w, label="Avant compression", color=[c + "CC" for c in colors])
bars2 = ax.bar([i + w/2 for i in x], df["perplexity_après"], w, label="Après compression", color=colors)
ax.set_xticks(x); ax.set_xticklabels(opts, rotation=15, ha='right')
ax.set_title("Perplexité (↓ meilleur)"); ax.set_ylabel("PPL")
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

# 2. Taille du modèle
ax = axes[1]
ax.bar([i - w/2 for i in x], df["taille_avant_MB"], w, label="Avant", color="#4C72B0AA")
ax.bar([i + w/2 for i in x], df["taille_après_MB"], w, label="Après", color="#4C72B0")
ax.set_xticks(x); ax.set_xticklabels(opts, rotation=15, ha='right')
ax.set_title("Taille du modèle (Mo)"); ax.set_ylabel("MB")
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

# 3. Ratio de compression
ax = axes[2]
bars = ax.bar(opts, df["compression_ratio"], color=colors)
for bar, v in zip(bars, df["compression_ratio"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{v:.2f}x",
            ha='center', va='bottom', fontsize=9)
ax.set_xticklabels(opts, rotation=15, ha='right')
ax.set_title("Ratio de compression (↑ meilleur)"); ax.set_ylabel("Ratio")
ax.grid(True, alpha=0.3, axis='y')

# 4. Latence avant/après
ax = axes[3]
ax.bar([i - w/2 for i in x], df["latence_avant_s"], w, label="Avant", color="#DD8452AA")
ax.bar([i + w/2 for i in x], df["latence_après_s"], w, label="Après", color="#DD8452")
ax.set_xticks(x); ax.set_xticklabels(opts, rotation=15, ha='right')
ax.set_title("Latence de génération (s) (↓ meilleur)"); ax.set_ylabel("Secondes")
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

# 5. Tokens/sec avant/après
ax = axes[4]
ax.bar([i - w/2 for i in x], df["tokens_par_sec_avant"], w, label="Avant", color="#55A868AA")
ax.bar([i + w/2 for i in x], df["tokens_par_sec_après"], w, label="Après", color="#55A868")
ax.set_xticks(x); ax.set_xticklabels(opts, rotation=15, ha='right')
ax.set_title("Débit (tokens/sec) (↑ meilleur)"); ax.set_ylabel("tok/s")
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

# 6. Temps d'entraînement
ax = axes[5]
bars = ax.bar(opts, df["temps_entraînement_s"] / 60, color=colors)
for bar, v in zip(bars, df["temps_entraînement_s"] / 60):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f"{v:.1f}m",
            ha='center', va='bottom', fontsize=9)
ax.set_xticklabels(opts, rotation=15, ha='right')
ax.set_title("Temps d'entraînement (min)"); ax.set_ylabel("Minutes")
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle("Comparaison des optimiseurs — DistilGPT-2 / TinyStories", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("results/comparison_charts.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Graphiques sauvegardés → results/comparison_charts.png")

---
## Section 7 — Exemples de génération avant/après compression

Comparaison qualitative de la génération de texte sur les mêmes prompts :
1. **Avant compression** : modèle fine-tuné float32
2. **Après compression** : modèle quantifié int8 (dynamic quantization)

Cette comparaison permet d'évaluer l'impact de la quantification sur la **qualité textuelle** au-delà des métriques numériques.

In [ ]:
import json, os, glob

example_files = glob.glob("results/text_examples_*.json")

for filepath in sorted(example_files):
    opt_name = os.path.basename(filepath).replace("text_examples_", "").replace(".json", "")
    with open(filepath, encoding="utf-8") as f:
        data = json.load(f)

    print(f"\n{'='*70}")
    print(f"  🔧 Optimiseur : {opt_name}")
    print(f"{'='*70}")

    for prompt in data.get("avant_compression", {}).keys():
        avant = data["avant_compression"].get(prompt, "N/A")
        après = data["après_compression"].get(prompt, "N/A")
        print(f"\n  📝 Prompt : \"{prompt}\"")
        print(f"  ┌─ AVANT  : {avant[:200]}")
        print(f"  └─ APRÈS  : {après[:200]}")

---
## Section 8 — Discussion et conclusions

### 8.1 Synthèse des résultats

Le tableau ci-dessous (généré automatiquement à partir des résultats) résume les compromis entre les 5 optimiseurs évalués :

In [ ]:
import pandas as pd
import json

df = pd.read_csv("results/resultats_comparatifs.csv")
with open("results/summary.json") as f:
    summary = json.load(f)

best = summary["meilleur_optimiseur"]
best_row = df[df["optimiseur"] == best].iloc[0]

# Métriques clés pour la discussion
ppl_degradation = df["perplexity_après"] - df["perplexity_avant"]
mean_ppl_degrad = ppl_degradation.mean()
mean_ratio = df["compression_ratio"].mean()
mean_size_before = df["taille_avant_MB"].mean()
mean_size_after = df["taille_après_MB"].mean()

# Meilleur pour l'Edge AI = meilleur compromis PPL + latence après compression
df["score_edge"] = df["perplexity_après"] * df["latence_après_s"]
best_edge = df.loc[df["score_edge"].idxmin(), "optimiseur"]

print("=" * 70)
print("  SYNTHÈSE AUTOMATIQUE DES RÉSULTATS")
print("=" * 70)
print(f"""
Optimiseurs testés    : {', '.join(df['optimiseur'].tolist())}
Meilleure perplexité  : {best} (PPL={best_row['perplexity_avant']:.2f})
Meilleur Edge AI      : {best_edge} (compromis PPL×latence après compression)

Quantification dynamique (int8) — impact moyen sur les 5 modèles :
  • Taille     : {mean_size_before:.1f} MB → {mean_size_after:.1f} MB  (×{mean_ratio:.2f})
  • PPL        : dégradation moyenne de +{mean_ppl_degrad:.2f} points
  • Latence    : voir tableau ci-dessus
""")
print("=" * 70)

### 8.2 Analyse du compromis compression / performance

#### Comparaison des optimiseurs

- **AdamW** : référence solide, convergence stable et prédictible. Performance de base attendue la plus élevée grâce à la correction du weight decay.
- **Adafactor** : optimiseur mémoire-efficace (pas de moments du second ordre complets), particulièrement adapté aux modèles larges. Performance comparable à AdamW sur les modèles compacts comme DistilGPT-2.
- **Lion** : optimiseur récent (Chen et al., 2023) basé sur le signe du gradient. Tend à nécessiter un LR plus faible mais peut surpasser AdamW sur des tâches de génération.
- **LAMB** : optimisé pour l'entraînement par large batch (adaptatif par layer). Sur de petits batch comme ici, l'avantage est limité.
- **SGD + momentum + Nesterov** : baseline classique. Souvent sous-performant sur les LLM en raison de sa sensibilité au LR initial et de l'absence d'adaptation.

#### Impact de la quantification dynamique int8

La quantification dynamique (post-training, sans recalibration) offre :
- **Réduction de taille** : facteur ~2× (float32 → int8 sur les couches Linear)
- **Accélération sur CPU** : les kernels int8 de PyTorch sont optimisés pour l'inférence CPU
- **Dégradation de perplexité** : faible (+quelques points en général), car DistilGPT-2 est robuste à la quantification

#### Recommandation pour l'Edge AI

Pour un déploiement sur dispositif contraint (mobile, embarqué) :
1. **Optimiseur recommandé** : AdamW ou Lion, selon les résultats obtenus
2. **Compression** : quantification dynamique int8 — rapport qualité/complexité optimal (pas de recalibration nécessaire, implémentation native PyTorch)
3. **Compromis** : la combinaison fine-tuning + quantification réduit la taille d'environ 50% avec une dégradation de perplexité inférieure à 5% dans la majorité des cas

### 8.3 Limites et perspectives

- **Quantification statique (PTQ)** : nécessite un dataset de calibration mais peut offrir un ratio de compression supérieur
- **Quantization-Aware Training (QAT)** : simule la quantification pendant l'entraînement pour réduire la dégradation
- **Pruning** : technique complémentaire pour réduire le nombre de paramètres actifs
- **Distillation** : entraîner un modèle encore plus petit (student) guidé par DistilGPT-2 (teacher)

In [ ]:
# Tableau récapitulatif final avec mise en forme
import pandas as pd

df = pd.read_csv("results/resultats_comparatifs.csv")

# Colonnes clés pour le rapport
cols_rapport = [
    "optimiseur",
    "taille_avant_MB", "taille_après_MB", "compression_ratio",
    "perplexity_avant", "perplexity_après",
    "latence_avant_s", "latence_après_s",
    "tokens_par_sec_avant", "tokens_par_sec_après",
    "mémoire_peak_GPU_GB", "temps_entraînement_s",
]

df_rapport = df[[c for c in cols_rapport if c in df.columns]]

# Highlight du meilleur par colonne
with open("results/summary.json") as f:
    summary = json.load(f)

print("📋 TABLEAU COMPARATIF FINAL")
print("=" * 120)
display(df_rapport.style
    .highlight_min(subset=["perplexity_avant", "perplexity_après", "latence_après_s"], color="lightgreen")
    .highlight_max(subset=["compression_ratio", "tokens_par_sec_après"], color="lightgreen")
    .format(precision=3)
)

print(f"\n🏆 Meilleur optimiseur (perplexité) : {summary['meilleur_optimiseur']}")
print(f"⏱  Temps GPU total : {summary['temps_total_GPU_min']:.1f} minutes")
print(f"💰 Coût estimé     : ~${summary['coût_estimé_USD']:.3f} USD")

---
## Annexe — Commandes utiles Modal

```bash
# Lancer le pipeline complet
modal run modal_distilgpt2.py

# Lancer un dry run (validation rapide, 0 crédit)
modal run modal_distilgpt2.py --dry-run

# Récupérer tous les résultats en local
modal volume get distilgpt2-results-v1 / ./results/ --force

# Lister le contenu du Volume
modal volume ls distilgpt2-results-v1 /

# Consulter les logs d'un run
modal app logs distilgpt2-edge-ai

# Voir l'utilisation de crédits
modal profile current
```